In [ ]:
import numpy as np
import scipy
import utils
import pandas as pd

In [ ]:
def get_atm_center(ds, sfc_center):
    nan_mask = ds['gh'].isnull()
    ds_atm = ds.where(~nan_mask, drop = True).isel(latitude = slice(50,-50), longitude = slice(50,-50))
    ds_atm = ds_atm.sel(latitude = slice(sfc_center['center_lat'].values - 0.5, sfc_center['center_lat'].values + 0.5),
                        longitude = slice(sfc_center['center_lon'].values - 0.5, sfc_center['center_lon'] + 0.5))
    
    

    hgt_threshold = np.quantile(ds_atm['gh'], 0.01)
    hgt_clusters = ds_atm['gh'].where(ds_atm['gh'] < hgt_threshold)
    ght_clusters_mask = ~hgt_clusters.isnull()

    clusters = scipy.ndimage.label(ght_clusters_mask)
    cluster_size = []
    for a in np.arange(clusters[1]):
        cluster_number = a + 1
        cluster_size.append(np.sum(np.where(clusters[0] == cluster_number, 1, 0)))

    
    largest_cluster = np.argmax(cluster_size) + 1
    center_y, center_x = scipy.ndimage.center_of_mass(clusters[0], largest_cluster)
    mslp = ds_atm['gh'].where(clusters[0] == largest_cluster).min().values

    center_lat = ds_atm.latitude.isel(latitude =int(np.round(center_y)))
    center_lon = ds_atm.longitude.isel(longitude =int(np.round(center_x)))
    
    center_info = dict(center_lat = center_lat, center_lon = center_lon, mslp = mslp)
    return center_info

def get_sfc_center(ds):
    min_pressure = ds['prmsl'].min().values

    ## Threshold is just based on the lowest mslp within the frame
    ## Given that the calculated center is the center of mass for 
    ## the largest contingous 1% low pressure area, final center
    ## pressure and lowest pressure in the frame may not be exactly
    ## the same.
    if min_pressure > 1005: # Pressure threshold (pa) 1005mb
        print(f'{min_pressure}mb is over the pressure threshold')
        return None
    nan_mask = ds['prmsl'].isnull()
    ds_sfc = ds.where(~nan_mask, drop = True).isel(latitude = slice(50,-50), longitude = slice(50,-50))
    

    slp_threshold = np.quantile(ds_sfc['prmsl'], 0.01)
    slp_clusters = ds_sfc['prmsl'].where(ds_sfc['prmsl'] < slp_threshold)
    slp_clusters_mask = ~slp_clusters.isnull()

    clusters = scipy.ndimage.label(slp_clusters_mask)
    cluster_size = []
    for a in np.arange(clusters[1]):
        cluster_number = a + 1
        cluster_size.append(np.sum(np.where(clusters[0] == cluster_number, 1, 0)))

    
    largest_cluster = np.argmax(cluster_size) + 1
    center_y, center_x = scipy.ndimage.center_of_mass(clusters[0], largest_cluster)
    mslp = ds_sfc['prmsl'].where(clusters[0] == largest_cluster).min()

    center_lat = ds_sfc.latitude.isel(latitude =int(np.round(center_y)))
    center_lon = ds_sfc.longitude.isel(longitude =int(np.round(center_x)))
    
    center_info = dict(center_lat = center_lat, center_lon = center_lon, mslp = mslp)
    return center_info

def random_link_selection(num_of_frames, links_path):
    result = []
    with open(links_path) as f:
        links = pd.read_csv(f)
    random_frames = links.sample(num_of_frames)
    for link in random_frames.values:
        result.append(('https://noaa-nws-hafs-pds.s3.amazonaws.com/' + link).squeeze())
    
    return result
    

In [ ]:
single_url = 'https://noaa-nws-hafs-pds.s3.amazonaws.com/hfsa/20230927/12/17l.2023092712.hfsa.storm.atm.f024.grb2'

atm_args = dict(typeOfKey = 'isobaricInhPa',filename="Model_Data/temp_gribs/atm_temp.grib2" )
sfc_args = dict(typeOfKey = 'meanSea',filename="Model_Data/temp_gribs/sfc_temp.grb2" )

In [74]:
random_links = random_link_selection(8, 'Model_Data\links\link_list.txt')

<>:1: SyntaxWarning: invalid escape sequence '\l'
<>:1: SyntaxWarning: invalid escape sequence '\l'
C:\Users\12ian\AppData\Local\Temp\ipykernel_20076\925719778.py:1: SyntaxWarning: invalid escape sequence '\l'
  random_links = random_link_selection(8, 'Model_Data\links\link_list.txt')


In [ ]:
random_links[0]

In [75]:
sfc_ds = utils.download_and_open(random_links[0], **sfc_args)
sfc_ds['prmsl'] = sfc_ds['prmsl']/100

Ignoring index file 'Model_Data/temp_gribs/sfc_temp.grb2.5b7b6.idx' older than GRIB file


In [76]:
sfc_ds

<xarray.Dataset> Size: 6MB
Dimensions:     (latitude: 801, longitude: 1001)
Coordinates:
  * latitude    (latitude) float64 6kB 5.295 5.315 5.335 ... 21.25 21.27 21.3
  * longitude   (longitude) float64 8kB 243.7 243.8 243.8 ... 263.7 263.7 263.7
    time        datetime64[ns] 8B ...
    step        timedelta64[ns] 8B ...
    meanSea     float64 8B ...
    valid_time  datetime64[ns] 8B ...
Data variables:
    prmsl       (latitude, longitude) float32 3MB nan nan nan ... nan nan nan
    mslet       (latitude, longitude) float32 3MB ...
Attributes:
    GRIB_edition:            2
    GRIB_centre:             kwbc
    GRIB_centreDescription:  US National Weather Service - NCEP
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             US National Weather Service - NCEP
    history:                 2026-03-10T14:18 GRIB to CDM+CF via cfgrib-0.9.1...

In [81]:
sfc_center = get_sfc_center(sfc_ds)

1004.6295


In [86]:
sfc_center['mslp'].values.squeeze()

array(1005.99896, dtype=float32)

In [87]:
atm_ds = utils.download_and_open(random_links[0], **atm_args)
atm_ds = atm_ds.sel(isobaricInhPa = [sfc_center['mslp'].values, 700.0, 200.0], method = 'nearest')

Ignoring index file 'Model_Data/temp_gribs/atm_temp.grib2.5b7b6.idx' older than GRIB file
skipping variable: paramId==3017 shortName='dpt'
Traceback (most recent call last):
  File "c:\Users\12ian\anaconda3\envs\ATMS523-Proj-Everything-Else\Lib\site-packages\cfgrib\dataset.py", line 726, in build_dataset_components
    dict_merge(variables, coord_vars)
    ~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\12ian\anaconda3\envs\ATMS523-Proj-Everything-Else\Lib\site-packages\cfgrib\dataset.py", line 642, in dict_merge
    raise DatasetBuildError(
    ...<2 lines>...
    )
cfgrib.dataset.DatasetBuildError: key present and new value is different: key='isobaricInhPa' value=Variable(dimensions=('isobaricInhPa',), data=array([1000.,  975.,  950.,  925.,  900.,  875.,  850.,  825.,  800.,
        775.,  750.,  725.,  700.,  675.,  650.,  625.,  600.,  575.,
        550.,  525.,  500.,  475.,  450.,  425.,  400.,  375.,  350.,
        325.,  300.,  275.,  250.,  225.,  200.,  175.,  150.,  125.